In [1]:
import sys
import os

# Get the directory of the current notebook
current_dir = os.path.dirname(os.path.abspath('__file__'))

# Move up to the root directory
root_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add the root directory to the sys.path
sys.path.append(root_dir)

from graph_gen.gen_graph import gen_graph
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain_openai.chat_models import ChatOpenAI
from langchain.agents import create_openai_functions_agent
from langgraph.prebuilt.tool_executor import ToolExecutor
from typing import Annotated, Dict, List, Optional, TypedDict, Tuple, Sequence, Union, Literal
from langchain_core.messages import AIMessage, BaseMessage, FunctionMessage, HumanMessage, ToolMessage
from langgraph.graph import END, StateGraph, MessageGraph
from langchain_core.agents import AgentAction, AgentFinish
from langchain.output_parsers.openai_tools import (
    JsonOutputToolsParser,
    PydanticToolsParser
)
from langchain.agents.output_parsers.openai_functions import OpenAIFunctionsAgentOutputParser
import operator
import re

### [LangGraph: Agent Executor](https://www.youtube.com/watch?v=9dXp5q3OFdQ&list=PLfaIDFEXuae16n2TWUkKq5PgJ0w6Pkwtg&index=2) <img src="https://upload.wikimedia.org/wikipedia/commons/4/42/YouTube_icon_%282013-2017%29.png" alt="YouTube" width="20" height="20">

In [2]:
# State
class AgentState(TypedDict):
    input: str
    chat_history: List[BaseMessage]
    agent_outcome: Union[AgentAction, AgentFinish, None]
    agent_scratchpad: List[object]
    intermediate_steps: Annotated[list[tuple[AgentAction, str]], operator.add]

In [3]:
# Nodes
def agent(state):
    agent_outcome = agent_runnable.invoke(state)
    return { "agent_outcome": agent_outcome }

def action(state):
    agent_action = state['agent_outcome']
    output = tool_executor.invoke(agent_action)
    return { "intermediate_steps": [(agent_action, str(output))] }

In [4]:
# Tools
tools = [TavilySearchResults(max_results=1)]
prompt = hub.pull("hwchase17/openai-functions-agent")
llm = ChatOpenAI(model="gpt-4o", streaming=True)
agent_runnable = prompt | llm.bind_tools(tools) | OpenAIFunctionsAgentOutputParser()
tool_executor = ToolExecutor(tools)

def print_result(s):
    if 'agent' in s and 'agent_outcome' in s['agent'] and isinstance(s['agent']['agent_outcome'], AgentFinish):
        print(s['agent']['agent_outcome'].return_values['output'])
    else:
        print(s)
    print()

/opt/anaconda3/envs/py312/lib/python3.12/site-packages/langchain_core/_api/beta_decorator.py:87: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  warn_beta(
/var/folders/cd/3w8prkld6nv0s9kb69pf_wy80000gn/T/ipykernel_5545/605323754.py:6: LangGraphDeprecationWarning: ToolExecutor is deprecated as of version 0.2.0 and will be removed in 0.3.0. Use langgraph.prebuilt.ToolNode instead.
  tool_executor = ToolExecutor(tools)


In [5]:
# Nodes
def init(state):
    return { "chat_history": [], "agent_outcome": None, "agent_scratchpad": [] }
    
# Graph
graph_spec = """

init(AgentState)
   => agent
   
agent
    is_finished => END
    => action

action
    => agent

"""

# Conditional edges
def is_finished(state):
    return isinstance(state['agent_outcome'], AgentFinish)

In [6]:
# Run the graph
# - generate the python langgraph code
# - execute the code, runnable graph in variable 'app'
# - test the graph
graph_code = gen_graph("agent_exec", graph_spec)
print(graph_code)
exec(graph_code)

agent_exec = StateGraph(AgentState)
agent_exec.add_node('init', init)
agent_exec.add_node('agent', agent)
agent_exec.add_node('action', action)

agent_exec.set_entry_point('init')

agent_exec.add_edge('init', 'agent')
def after_agent(state: AgentState):
    if is_finished(state):
        return 'END'
    return 'action'

agent_dict = {'END': END, 'action': 'action'}
agent_exec.add_conditional_edges('agent', after_agent, agent_dict)

agent_exec.add_edge('action', 'agent')

agent_exec = agent_exec.compile()


In [7]:
# Another test
inputs = { "input": "who is a female voice actress who does animal voices", "chat_history": [], "agent_scratchpad": [] }
for s in agent_exec.stream(inputs):
    print_result(s)
inputs = { "input": "who is a female voice actress who does animal voices", "chat_history": [], "agent_scratchpad": [] }
result1 = agent_exec.invoke(inputs)
print("RESULT1", result1)
inputs = { "input": "who is a female voice actress who does animal voices", "agent_scratchpad": [] }
result2 = agent_exec.invoke(inputs)
print("RESULT2", result2)

{'init': {'chat_history': [], 'agent_outcome': None, 'agent_scratchpad': []}}



RESULT1 {'input': 'who is a female voice actress who does animal voices', 'chat_history': [], 'agent_outcome': AgentFinish(return_values={'output': ''}, log=''), 'agent_scratchpad': [], 'intermediate_steps': []}
RESULT2 {'input': 'who is a female voice actress who does animal voices', 'chat_history': [], 'agent_outcome': AgentFinish(return_values={'output': ''}, log=''), 'agent_scratchpad': [], 'intermediate_steps': []}


In [8]:
result1

{'input': 'who is a female voice actress who does animal voices',
 'chat_history': [],
 'agent_outcome': AgentFinish(return_values={'output': ''}, log=''),
 'agent_scratchpad': [],
 'intermediate_steps': []}

In [9]:
result2

{'input': 'who is a female voice actress who does animal voices',
 'chat_history': [],
 'agent_outcome': AgentFinish(return_values={'output': ''}, log=''),
 'agent_scratchpad': [],
 'intermediate_steps': []}